In [ ]:
import json

import openeo.processes
from openeo.rest.udp import build_process_dict
from utils import udp_params, urls, utils

import openeo

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

In [ ]:
connection.authenticate_oidc()

# Parameters

In [ ]:
upstream_and_downstream = udp_params.UPSTREAM_AND_DOWNSTREAM

In [ ]:
temporal_extent = udp_params.TEMPORAL_EXTENT

In [ ]:
baseline_start = udp_params.BASELINE_START
baseline_end = udp_params.BASELINE_END

In [ ]:
water_occurrence_threshold = udp_params.WATER_OCCURRENCE_THRESHOLD  # units: %

In [ ]:
# for a pixel to be considered anomalous,
# its anomaly must be >= anomalyZThreshold
# and its value must be >= effectThreshold * climatological median

anomaly_z_threshold = udp_params.ANOMALY_Z_THRESHOLD
effect_threshold = udp_params.EFFECT_THRESHOLD

In [ ]:
# number of months in a year that must be anomalous to be a hotspot
min_months_per_year_for_hotspot = udp_params.MIN_MONTHS_PER_YEAR_FOR_HOTSPOT

# min connected area of hotspot pixels (8-connectivity)
min_contiguous_pixels = udp_params.MIN_CONTIGUOUS_PIXELS

In [ ]:
parameters = [
    upstream_and_downstream,
    temporal_extent,
    baseline_start,
    baseline_end,
    water_occurrence_threshold,
    anomaly_z_threshold,
    effect_threshold,
    min_months_per_year_for_hotspot,
    min_contiguous_pixels,
]

# UDP

Sentinel 2 L2A

In [ ]:
optical_bands = ["B03", "B04", "B05"]

In [ ]:
S2 = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=upstream_and_downstream,
    temporal_extent=temporal_extent,
    bands=optical_bands + ["SCL"],
)

mask Sentinel 2 based on SCL

In [ ]:
scl = S2.band("SCL")
scl_mask = (
    (scl == 3)  # cloud shadow
    | (scl == 8)  # medium probability cloud
    | (scl == 9)  # high probability cloud
    | (scl == 10)  # cirrus
    | (scl == 11)  # snow or ice
)

In [ ]:
S2 = S2.filter_bands(optical_bands).mask(scl_mask)

persistent water mask

In [ ]:
gsw = connection.load_stac(
    urls.JRC_GSW_STAC_COLLECTION,
    spatial_extent=upstream_and_downstream,
    bands=["occurrence"],
)

In [ ]:
# drop time dimension (length 1)
gsw = gsw.drop_dimension("t")

In [ ]:
# water_extent_mask = gsw.band("occurrence") >= water_occurrence_threshold

# cannot use band math with Parameters 🙁
# https://forum.dataspace.copernicus.eu/t/udp-parameter-not-applied/5282

gte_water_occurrence_threshold = openeo.UDF.from_file(
    "../udf/binary_operator.py",
    runtime="Python",
    version="3.11",
    context={
        "operator": "gte",
        "argument": water_occurrence_threshold,
    },
)

water_extent_mask = (
    gsw.apply(gte_water_occurrence_threshold)
    .convert_data_type("bool")
    .band("occurrence")
)

In [ ]:
# reproject and resample onto S2
water_extent_mask = water_extent_mask.resample_cube_spatial(S2)

In [ ]:
# invert after resampling to avoid edge effects
inverse_water_extent_mask = utils.logical_not(water_extent_mask)

mask Sentinel 2 based on persistent water mask

In [ ]:
S2 = S2.mask(inverse_water_extent_mask)

computes indicies

- green = B03
- red = B04
- red_edge = B05

NDTI = (red - green) / (red + green)

NDCI = (red_edge - red) / (red_edge + red)

In [ ]:
green = S2.band("B03")
red = S2.band("B04")
red_edge = S2.band("B05")
ndti = red.normalized_difference(green).add_dimension("bands", "NDTI", type="bands")
ndci = red_edge.normalized_difference(red).add_dimension("bands", "NDCI", type="bands")
indices = ndti.merge_cubes(ndci)

monthly median composite

In [ ]:
monthly_median_composite = indices.aggregate_temporal_period(
    period="month",
    reducer=openeo.processes.median,
)

Climatology

median and standard deviation

In [ ]:
climatology_udf_context = {
    # NDTI is used as the primary variable for baseline construction,
    # anomaly detection and annual hotspot generation
    "target_band": "NDTI",
    "baseline_start": baseline_start,
    "baseline_end": baseline_end,
    "std_min": 0.0001,
}

climatology_udf = openeo.UDF.from_file(
    "./udf/monthly_climatology.py",
    runtime="Python",
    version="3.11",
    context=climatology_udf_context,
)

In [ ]:
climatology = monthly_median_composite.apply_dimension(
    process=climatology_udf,
    dimension="t",
)

In [ ]:
# ⚠️ openEO python client does not run UDF apply_metadata()!
# so we have to do it manually here

import udf.monthly_climatology

climatology.metadata = udf.monthly_climatology.apply_metadata(
    climatology.metadata, climatology_udf_context
)

calculate anomalies

In [ ]:
Z = (climatology.band("NDTI") - climatology.band("NDTI_median")) / climatology.band(
    "NDTI_std"
)

n_valid_months

For each year, count the number of months in which we have a valid value for Z.

ie. that month was not (cloud) masked.

In [ ]:
n_valid_months = Z.aggregate_temporal_period(
    period="year",
    reducer=openeo.processes.count,
)

exceedance event in month

- Z >= `anomalyZThreshold`
- X >= (1 + `effectThreshold`) * median

In [ ]:
# merge the climatology back in, so that we can do band math
Z = Z.add_dimension("bands", "NDTI_Z", type="bands")
Z = Z.merge_cubes(climatology)

In [ ]:
# does not propagate nodata
# pixels previously masked by `inverse_water_extent_mask` are set to 0
# exceedance = (Z.band("NDTI_Z") >= anomalyZThreshold) & (
#     Z.band("NDTI") >= (1 + effectThreshold) * Z.band("NDTI_median")
# )

# cannot use band math with Parameters 🙁
# https://forum.dataspace.copernicus.eu/t/udp-parameter-not-applied/5282

gte_anomaly_z = openeo.UDF.from_file(
    "../udf/binary_operator.py",
    runtime="Python",
    version="3.11",
    context={
        "operator": "gte",
        "argument": anomaly_z_threshold,
    },
)

anomaly_ok = Z.band("NDTI_Z").apply(gte_anomaly_z).convert_data_type("bool")

gte_effect_threshold = openeo.UDF.from_file(
    "../udf/binary_operator.py",
    runtime="Python",
    version="3.11",
    context={
        "operator": "gte",
        "argument": effect_threshold,
    },
)

effect_ok = (
    (Z.band("NDTI") / Z.band("NDTI_median") - 1.0)
    .apply(gte_effect_threshold)
    .convert_data_type("bool")
)

# these are now 2 separate datacubes
# so we have to stack them before combining

anomaly_ok = anomaly_ok.add_dimension("bands", "anomaly_ok", type="bands")
effect_ok = effect_ok.add_dimension("bands", "effect_ok", type="bands")

stacked_datacube = anomaly_ok.merge_cubes(effect_ok)
exceedance = stacked_datacube.band("anomaly_ok") & stacked_datacube.band("effect_ok")

exceedance_count = yearly sum of montly exceedance events

In [ ]:
# ⚠️ https://forum.dataspace.copernicus.eu/t/aggregate-temporal-period-with-sum-behaves-like-xor/5667
# exceedance has dtype bool: we can't sum that

exceedance_count = (1.0 * exceedance).aggregate_temporal_period(
    period="year",
    reducer=openeo.processes.sum,
)

A pixel is a hotspot (in a given year) if exceedance_count >= minMonthsPerYearForHotspot

In [ ]:
# hotspot = exceedance_count >= minMonthsPerYearForHotspot

# cannot use band math with Parameters 🙁
# https://forum.dataspace.copernicus.eu/t/udp-parameter-not-applied/5282

gte_min_months_per_year = openeo.UDF.from_file(
    "../udf/binary_operator.py",
    runtime="Python",
    version="3.11",
    context={
        "operator": "gte",
        "argument": min_months_per_year_for_hotspot,
    },
)

hotspot = exceedance_count.apply(gte_min_months_per_year).convert_data_type("bool")

apply connected component constraint minContiguousPixels

In [ ]:
connectivity_filter_udf_context = {
    "min_pixels": min_contiguous_pixels,
}

connectivity_filter_udf = openeo.UDF.from_file(
    "./udf/connectivity_filter.py",
    runtime="Python",
    version="3.11",
    context=connectivity_filter_udf_context,
)

In [ ]:
# datacube returned from UDF has float dtype

hotspot = hotspot.apply_neighborhood(
    process=connectivity_filter_udf,
    size=[
        {"dimension": "x", "value": 256, "unit": "px"},
        {"dimension": "y", "value": 256, "unit": "px"},
    ],
    # overlap needs to be big enough the reasonably allow for min_pixels
    overlap=[
        {"dimension": "x", "value": 8, "unit": "px"},
        {"dimension": "y", "value": 8, "unit": "px"},
    ],
)

hotspot_exceedance_count = exceedance_count only for hotspot pixels

In [ ]:
inverse_hotspot = utils.logical_not(hotspot)

In [ ]:
hotspot_exceedance_count = exceedance_count.mask(inverse_hotspot)

Generate the water_valid layer (as in the GEE reference implementation).

Defined as any pixel where we have a least 1 month of data in that year.

In [ ]:
water_valid = n_valid_months > 0

create the final raster, with bands:

- hotspot (pixel mask)
- hotspot_exceedance_count (0-12, masked)
- n_valid_months (0–12, masked)
- water_valid (pixel mask)

This raster has dtype float.

In [ ]:
b1 = hotspot.add_dimension("bands", "hotspot", type="bands")
b2 = hotspot_exceedance_count.add_dimension(
    "bands", "hotspot_exceedance_count", type="bands"
)
b3 = n_valid_months.add_dimension("bands", "n_valid_months", type="bands")
b4 = water_valid.add_dimension("bands", "water_valid", type="bands")
final_raster = b1.merge_cubes(b2).merge_cubes(b3).merge_cubes(b4)

aggregate over upstream and downstream geometries

count the number of hotspot pixels and the number of water_valid pixels per year.

In [ ]:
def reducer(data: openeo.processes.ProcessBuilder) -> openeo.processes.ProcessBuilder:
    """sum, count, mean"""
    return openeo.processes.array_create(
        [
            data.sum(),
            data.count(),
            data.mean(),
        ]
    )


vector_stats = final_raster.filter_bands(
    ["hotspot", "hotspot_exceedance_count", "water_valid"]
).aggregate_spatial(
    geometries=upstream_and_downstream,
    reducer=reducer,
)

# Serialise UDP

In [ ]:
summary = (
    "Detect NDTI turbidity hotspots from Sentinel-2, then aggregate "
    "yearly hotspot and water-coverage statistics for upstream control zone and "
    "downstream influence zone."
)
description = (
    "Computes monthly median Normalized Difference Turbidity Index (NDTI) "
    "from Sentinel-2 L2A. "
    "Persistent water is masked with Joint Research Centre (JRC) Global Surface Water (GSW). "
    "A baseline climatology (median and std) drives per-month exceedance tests. "
    "Pixels with enough exceedance months per year become hotspots. "
    "Small 8-connected components are removed. "
    "Returns a vector cube with bands: "
    "sum(hotspot), count(hotspot), avg(hotspot), "
    "sum(hotspot_exceedance_count), count(hotspot_exceedance_count), avg(hotspot_exceedance_count), "
    "sum(water_valid), count(water_valid), avg(water_valid). "
    "WARNING: depending on the requested download format, bands may be named differently - "
    "see: https://forum.dataspace.copernicus.eu/t/inconsistent-band-names-csv-parquet/5683."
)

udp_spec = build_process_dict(
    vector_stats,
    process_id="ndti_hotspots",
    summary=summary,
    description=description,
    parameters=parameters,
    returns={
        "description": "A vector cube",
        "schema": {
            "type": "object",
            "subtype": "datacube",  # vector-cube is deprecated
            "dimensions": [
                {"type": "geometry"},
                {"type": "bands"},
                {"type": "temporal", "axis": "t"},
            ],
        },
    },
)

In [ ]:
with open("udp.json", "w") as f:
    json.dump(udp_spec, f, indent=2)